<a href="https://colab.research.google.com/github/ekonishi8524/my-data-science-projects/blob/main/02_corona_and_nyc_taxis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
import pandas as pd

auth.authenticate_user()
print("Authentication successful!")

project_id = "YOUR_GOOGLE_CLOUD_PROJECT_ID"

query_2_1 = """
WITH taxi_base AS (
  SELECT
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS day_of_week,
    EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
    trip_distance,
    fare_amount,
    tip_amount,
    ROW_NUMBER() OVER(
      PARTITION BY EXTRACT(DAYOFWEEK FROM pickup_datetime), \
      EXTRACT(HOUR FROM pickup_datetime)
      ORDER BY trip_distance DESC
    ) AS distance_rank
  FROM
    `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2020`
  WHERE
    pickup_datetime BETWEEN '2020-01-01' AND '2020-12-31'
    AND passenger_count > 0
    AND trip_distance > 0
    AND fare_amount > 0
)
SELECT
  day_of_week,
  pickup_hour,
  COUNT(*) AS total_trips,
  ROUND(AVG(trip_distance), 2) AS avg_distance,
  ROUND(AVG(SAFE_DIVIDE(tip_amount, fare_amount)) * 100, 2) AS avg_tip_rate
FROM
  taxi_base
WHERE
  TRUE
GROUP BY
  day_of_week,
  pickup_hour
ORDER BY
  day_of_week,
  pickup_hour;
"""

df = pd.read_gbq(query_2_1, project_id=project_id, dialect="standard")

print(f"Shape of Acquired Data: {df.shape}")
df.head()

In [ ]:
import pandas as pd

day_map = {1: 'Sun', 2: 'Mon', 3: 'Tue', 4: 'Wed', 5: 'Thu', 6: 'Fri', 7: 'Sat'}
df['day_name'] = df['day_of_week'].map(day_map)

df['day_name'] = pd.Categorical(df['day_name'], categories=['Mon', 'Tue', 'Wed',\
                                                            'Thu', 'Fri', 'Sat', 'Sun'],\
                                 ordered=True)

pivot_df = df.pivot(index='day_name', columns='pickup_hour', values='avg_tip_rate')

print(pivot_df.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

pivot_df = pivot_df.apply(pd.to_numeric, errors='coerce')

plt.figure(figsize=(14, 6))

sns.heatmap(
    pivot_df,
    cmap='YlGnBu',
    annot=True,
    fmt=".1f",
    linewidths=.5,
    cbar_kws={'label': 'Average Tip Rate (%)'}
)

plt.title('NYC Taxi Average Tip Rate by Day and Hour (2020)', fontsize=16, pad=15)
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Day of Week', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
sat_3am = df[(df['day_of_week'] == 7) & (df['pickup_hour'] == 3)]
print(sat_3am[['total_trips', 'avg_tip_rate']])

In [ ]:
query_2_2 = """
SELECT
  EXTRACT(MONTH FROM pickup_datetime) AS pickup_month,
  EXTRACT(DAYOFWEEK FROM pickup_datetime) AS day_of_week,
  EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
  COUNT(*) AS total_trips,
  ROUND(AVG(SAFE_DIVIDE(tip_amount, fare_amount)) * 100, 2) AS avg_tip_rate,
  ROUND(AVG(trip_distance), 2) AS avg_distance
FROM
  `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2020`
WHERE
  pickup_datetime BETWEEN '2020-01-01' AND '2020-12-31'
  AND passenger_count > 0
  AND trip_distance > 0
  AND fare_amount > 0
GROUP BY
  pickup_month,
  day_of_week,
  pickup_hour
ORDER BY
  pickup_month,
  day_of_week,
  pickup_hour;
"""

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_monthly = pd.read_gbq(query_2_2, project_id=project_id, dialect="standard")

sat_3am_monthly = df_monthly[(df_monthly['day_of_week'] == 7) &\
                             (df_monthly['pickup_hour'] == 3)].copy()

sat_3am_monthly = sat_3am_monthly.sort_values('pickup_month')

print(sat_3am_monthly[['pickup_month', 'total_trips', 'avg_tip_rate']])

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

color = 'tab:blue'
ax1.set_xlabel('Month (2020)', fontsize=12)
ax1.set_ylabel('Average Tip Rate (%)', color=color, fontsize=12)
line1 = ax1.plot(sat_3am_monthly['pickup_month'], sat_3am_monthly['avg_tip_rate'],
                 color=color, marker='o', linewidth=2, label='Avg Tip Rate (%)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(range(1, 13))

ax2 = ax1.twinx()
color = 'tab:gray'
ax2.set_ylabel('Total Trips', color=color, fontsize=12)
line2 = ax2.plot(sat_3am_monthly['pickup_month'], sat_3am_monthly['total_trips'],
                 color=color, marker='x', linestyle='--', alpha=0.6, label='Total Trips')
ax2.tick_params(axis='y', labelcolor=color)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right')

plt.title('NYC Taxi Sat 3AM: Monthly Trends of Tip Rate & Trips in 2020',\
          fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
query_2_3 = """
SELECT
  EXTRACT(MONTH FROM pickup_datetime) AS pickup_month,
  EXTRACT(DAYOFWEEK FROM pickup_datetime) AS day_of_week,
  EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
  COUNT(*) AS total_trips,
  ROUND(AVG(SAFE_DIVIDE(tip_amount, fare_amount)) * 100, 2) AS avg_tip_rate
FROM
  `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2020`
WHERE
  pickup_datetime BETWEEN '2020-01-01' AND '2020-12-31'
  AND passenger_count > 0
  AND trip_distance > 0
  AND tip_amount <= fare_amount
GROUP BY
  pickup_month, day_of_week, pickup_hour
ORDER BY
  pickup_month, day_of_week, pickup_hour;
"""

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_monthly2 = pd.read_gbq(query_2_3, project_id=project_id, dialect="standard")

sat_3am_monthly2 = df_monthly2[(df_monthly2['day_of_week'] == 7) &\
                               (df_monthly2['pickup_hour'] == 3)].copy()

sat_3am_monthly2 = sat_3am_monthly2.sort_values('pickup_month')

print(sat_3am_monthly2[['pickup_month', 'total_trips', 'avg_tip_rate']])

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

color = 'tab:blue'
ax1.set_xlabel('Month (2020)', fontsize=12)
ax1.set_ylabel('Average Tip Rate (%)', color=color, fontsize=12)
line1 = ax1.plot(sat_3am_monthly2['pickup_month'], sat_3am_monthly2['avg_tip_rate'],
                 color=color, marker='o', linewidth=2, label='Avg Tip Rate (%)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(range(1, 13))

ax2 = ax1.twinx()
color = 'tab:gray'
ax2.set_ylabel('Total Trips', color=color, fontsize=12)
line2 = ax2.plot(sat_3am_monthly2['pickup_month'], sat_3am_monthly2['total_trips'],
                 color=color, marker='x', linestyle='--', alpha=0.6, label='Total Trips')
ax2.tick_params(axis='y', labelcolor=color)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right')

plt.title('NYC Taxi Sat 3AM: Monthly Trends of Tip Rate & Trips in 2020',\
          fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
query_2_4 = """
WITH jfk_trips AS (
  SELECT
    pickup_datetime,
    LAG(pickup_datetime) OVER(
      ORDER BY pickup_datetime
    ) AS prev_pickup_datetime
  FROM
    `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2020`
  WHERE
    pickup_datetime BETWEEN '2020-01-01' AND '2020-12-31'
    AND pickup_location_id = '132'
    AND passenger_count > 0
),
calculated_intervals AS (
  SELECT
    EXTRACT(MONTH FROM pickup_datetime) AS pickup_month,
    TIMESTAMP_DIFF(pickup_datetime, prev_pickup_datetime, MINUTE) AS interval_minutes
  FROM
    jfk_trips
  WHERE
    prev_pickup_datetime IS NOT NULL
)
SELECT
  pickup_month,
  COUNT(*) AS total_trips,
  ROUND(AVG(interval_minutes), 1) AS avg_interval_minutes
FROM
  calculated_intervals
WHERE
  interval_minutes BETWEEN 0 AND 1440
GROUP BY
  pickup_month
ORDER BY
  pickup_month;
"""

df_interval = pd.read_gbq(query_2_4, project_id=project_id, dialect="standard")

plt.figure(figsize=(10, 5))

plt.plot(df_interval['pickup_month'], df_interval['avg_interval_minutes'],
         marker='o', color='darkorange', linewidth=2)

plt.title('JFK Airport: Average Taxi Departure Interval (2020)', fontsize=14)
plt.xlabel('Month (2020)', fontsize=12)
plt.ylabel('Average Interval (Minutes)', fontsize=12)
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()